# 05 Backtest and Regime Analysis\n\nGoal: compare behavior across major market periods and outline a simple long/flat signal test.

In [ ]:
from pathlib import Path\nimport pandas as pd\n\nPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()\nDATA_PATH = PROJECT_ROOT / "data" / "processed" / "daily_sentiment.csv"

In [ ]:
daily = pd.read_csv(DATA_PATH, parse_dates=["Date"])\n\ndef assign_regime(date):\n    if pd.Timestamp("2008-01-01") <= date <= pd.Timestamp("2009-12-31"):\n        return "2008-2009 financial crisis"\n    if pd.Timestamp("2020-01-01") <= date <= pd.Timestamp("2020-12-31"):\n        return "2020 COVID shock"\n    if pd.Timestamp("2022-01-01") <= date <= pd.Timestamp("2023-12-31"):\n        return "2022-2023 rate-hike cycle"\n    return "Other"\n\ndaily["regime"] = daily["Date"].apply(assign_regime)\nregime_summary = daily.groupby("regime").agg(\n    days=("Date", "count"),\n    avg_headlines=("headline_count", "mean"),\n    avg_sentiment=("sentiment_compound", "mean"),\n    avg_next_return=("return_next_day", "mean"),\n    volatility=("return_next_day", "std"),\n    up_day_rate=("direction_next_day", "mean"),\n)\nregime_summary

In [ ]:
# Placeholder long/flat signal: long when compound sentiment is positive.\n# Improve this later by using model probabilities and transaction costs.\ndaily["signal_long"] = (daily["sentiment_compound"] > 0).astype(int)\ndaily["strategy_return"] = daily["signal_long"] * daily["return_next_day"]\n\nbacktest_summary = daily.groupby("regime").agg(\n    market_return=("return_next_day", "sum"),\n    strategy_return=("strategy_return", "sum"),\n    avg_signal=("signal_long", "mean"),\n)\nbacktest_summary